In [ ]:
short_names = {
    'Pòlissa/Póliza/Policy': 'POLICY',
    'Diàmetre comptador (cm)/Diámetro contador (cm)/Counter diameter (cm)': 'DIAMETER',
    'Data/Fecha/Date': 'HOUR/DATE',
    'Índex de lectura (L/h)/Índice de lectura (L/h)/Reading index (L/h)': 'CONSUMPTION',
}

In [ ]:
import pyarrow.dataset as ds
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
import pandas as pd

file_path = '../data/lectures_horaries_ABD.parquet'

dataset = ds.dataset(file_path, format="parquet")
table = dataset.to_table()
df = table.to_pandas()
df_copy = table.to_pandas()

In [ ]:
df_copy = df_copy.rename(columns=short_names)
# Convert 'Data/Fecha/Date' column to datetime format
df_copy['HOUR/DATE'] = pd.to_datetime(df_copy['HOUR/DATE'])

df_copy['DATE'] = df_copy['HOUR/DATE'].dt.date
df_copy['HOUR'] = df_copy['HOUR/DATE'].dt.hour

df_copy = df_copy.drop(columns=['HOUR/DATE'])

In [ ]:
label_encoders = {}
for col in df_copy.select_dtypes(include='object').columns:  # 'object' dtype selects categorical columns
    le = LabelEncoder()
    df_copy[col] = le.fit_transform(df_copy[col])
    label_encoders[col] = le

In [ ]:
# Sort the DataFrame by 'POLICY' and 'DATE' to ensure correct diff calculation
df_copy = df_copy.sort_values(by=['POLICY', 'DATE', 'HOUR'])

# Calculate the differential for each identifier separately
df_copy['FLOW'] = df_copy.groupby('POLICY')['CONSUMPTION'].diff()

# Debugging: Check if 'FLOW' column is created
print("Columns after creating 'FLOW':", df_copy.columns)

In [ ]:
# Drop rows with NaN values in 'FLOW'
df_copy = df_copy.dropna(subset=["FLOW"])

# Debugging: Check if 'FLOW' column still exists after dropping NaNs
print("Columns after dropping NaNs in 'FLOW':", df_copy.columns)

In [ ]:
# Select features and target variable
features = ['POLICY', 'DIAMETER','CONSUMPTION', 'DATE', 'HOUR']
target = 'FLOW'

X = df_copy[features]
y = df_copy[target]

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.16, random_state=42)

In [ ]:
# Initialize the gradient boosting regressor
gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)

In [ ]:
# Train the model
gbr.fit(X_train, y_train)

In [ ]:
# Make predictions
y_pred = gbr.predict(X_test)

In [ ]:
# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")

In [ ]:
# Plot the feature importances
plt.figure(figsize=(10, 6))
feature_importances = gbr.feature_importances_
sns.barplot(x=feature_importances, y=features)
plt.title("Feature Importances")
plt.show()